<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>Basic Agentic Harness</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [1]:
from __future__ import annotations

import json
import re
import time
from dataclasses import dataclass, field
from typing import Any, Callable

import anthropic
import matplotlib.pyplot as plt
import watermark

%load_ext watermark
%matplotlib inline

Matplotlib is building the font cache; this may take a moment.


We start by printing out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.13.13
IPython version      : 9.12.0

Compiler    : Clang 22.1.3 
OS          : Darwin
Release     : 25.5.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: 7acbca3b06ff9ca59335e165e926a14061ee8622

anthropic : 0.96.0
json      : 2.0.9
matplotlib: 3.10.8
re        : 2.2.1
watermark : 2.6.0



Load default figure style

In [3]:
plt.style.use('d4sci.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

## An LLM provider abstraction

We want to swap backends easily (Anthropic today, OpenAI tomorrow, a local model next week) *and* have the notebook run without any API key. The cleanest way is a tiny protocol: one method, `complete(system, user)`, that takes two strings and returns one.

In [4]:
class AnthropicProvider:
    """Thin wrapper around the Anthropic Messages API."""

    def __init__(self, model: str = "claude-sonnet-4-5-20250929"):
        self._client = anthropic.Anthropic()
        self._model = model

    def complete(self, system: str, user: str) -> str:
        response = self._client.messages.create(
            model=self._model,
            max_tokens=1024,
            system=system,
            messages=[{"role": "user", "content": user}],
        )
        # Concatenate all text blocks
        return "".join(b.text for b in response.content if b.type == "text")


In [5]:
llm = AnthropicProvider()

## Core state

Everything the agent "knows" at any moment lives in a single state object. A useful mental checklist:

| Field | What it holds | Example |
|---|---|---|
| **goal** | user objective + acceptance criteria | *"Compute the pop. % of the capital of France"* |
| **trace** | step-by-step log of thoughts, actions, observations | *list of turns* |
| **memory** | key facts the agent has discovered | *{"capital": "Paris"}* |
| **budget** | remaining steps / tokens / tool calls | *{"max_steps": 10, "used": 3}* |
| **status** | `running` / `done` / `failed` | `running` |

Keeping all of this in *one typed object* (rather than a bag of variables) will make our lives easier when things go wrong.

In [6]:
@dataclass
class Step:
    """One iteration of the control loop."""
    index: int
    thought: str
    action: dict            # {"type": "tool_call", ...} or {"type": "final", ...}
    observation: Any        # whatever the tool returned
    latency_ms: int

@dataclass
class Budget:
    max_steps: int = 10
    max_tool_calls: int = 15
    steps_used: int = 0
    tool_calls_used: int = 0

    def has_room(self) -> bool:
        return (
            self.steps_used < self.max_steps
            and self.tool_calls_used < self.max_tool_calls
        )

@dataclass
class AgentState:
    goal: str
    trace: list[Step] = field(default_factory=list)
    memory: dict[str, Any] = field(default_factory=dict)
    budget: Budget = field(default_factory=Budget)
    status: str = "running"          # running | done | failed
    final_answer: str | None = None


## Tools

A tool is just a Python function, plus two pieces of metadata:

1. A **description** the LLM will read to decide whether to call it
2. A **parameter schema** so we can validate the args before executing

We'll keep the registry dead simple: a dict that maps tool names to a `Tool` dataclass.

In [7]:
@dataclass
class Tool:
    name: str
    description: str
    params: dict[str, type]      # param_name -> expected Python type
    fn: Callable[..., Any]

    def schema_str(self) -> str:
        """One-line description the LLM sees."""
        args = ", ".join(f"{k}: {v.__name__}" for k, v in self.params.items())
        return f"{self.name}({args}) — {self.description}"

    def validate(self, args: dict) -> str | None:
        """Return None if valid, an error message otherwise."""
        missing = set(self.params) - set(args)
        if missing:
            return f"Missing args: {sorted(missing)}"
        for k, v in args.items():
            if k not in self.params:
                return f"Unknown arg: {k}"
            if not isinstance(v, self.params[k]):
                return (
                    f"Arg {k!r} expected {self.params[k].__name__}, "
                    f"got {type(v).__name__}"
                )
        return None


Now we define two toy tools the agent can use.

- `lookup` returns canned facts (standing in for a search tool)
- `calculator` evaluates a simple arithmetic expression

Note the calculator: we use `eval` but first require the expression to match a whitelist of safe characters (digits, arithmetic operators, parentheses, dots, whitespace). In production you'd use a proper expression parser — `eval` is a loaded footgun. For teaching, the whitelist makes the risk visible.

In [8]:
# --- Mock knowledge base for the lookup tool ---
_KB = {
    "capital of france": "Paris",
    "paris population": "2.1 million",
    "eiffel tower height": "330 meters",
}

def lookup(query: str) -> str:
    key = query.lower().strip()
    return _KB.get(key, f"(no result for {query!r})")


def calculator(expression: str) -> float:
    # Only allow digits, operators, parens, dots, and whitespace.
    if not re.fullmatch(r"[0-9+\-*/().\s]+", expression):
        raise ValueError(f"Unsafe expression: {expression!r}")
    return eval(expression, {"__builtins__": {}}, {})

### Tool Registry

The tool registry is where we keep track of what tools the LLM is allowed to use.
We make it as simple as possible: just a dictionary mapping names to `Tool` objects as defined above

In [9]:
TOOLS: dict[str, Tool] = {
    "lookup": Tool(
        name="lookup",
        description="Look up a fact by natural-language query.",
        params={"query": str},
        fn=lookup,
    ),
    "calculator": Tool(
        name="calculator",
        description="Evaluate an arithmetic expression. Supports + - * / and parentheses.",
        params={"expression": str},
        fn=calculator,
    ),
}

## The planner prompt

On every iteration we send the LLM:

- The **system prompt** — its role, the tools available, the output format
- The **user prompt** — the goal plus the trace so far

We ask for a strict JSON response: either a tool call or a final answer. The schema is what makes the loop robust. Without it, we'd be parsing free-form English, which is brittle.

In [10]:
SYSTEM_PROMPT = """You are a helpful agent that solves problems by calling tools.

You MUST respond with a single JSON object in one of these two shapes:

1. Tool call:
{"thought": "...", "action": {"type": "tool_call", "name": "<tool>", "args": {...}}}

2. Final answer:
{"thought": "...", "action": {"type": "final", "answer": "..."}}

Available tools:
<<TOOL_LIST>>

Rules:
- Output JSON only, no markdown, no backticks, no prose.
- Use a tool when you need external information or computation.
- Produce a final answer only when you have enough evidence.
"""

In [11]:
def build_system_prompt() -> str:
    tool_list = "\n".join(f"- {t.schema_str()}" for t in TOOLS.values())
    return SYSTEM_PROMPT.replace("<<TOOL_LIST>>", tool_list)


def build_user_prompt(state: AgentState) -> str:
    parts = [f"Goal: {state.goal}", ""]
    if state.trace:
        parts.append("Trace so far:")
        for step in state.trace:
            parts.append(f"  Step {step.index}: thought={step.thought}")
            parts.append(f"    action={json.dumps(step.action)}")
            parts.append(f"    observation={step.observation}")
        parts.append("")
    parts.append("What should we do next? Respond with JSON only.")
    return "\n".join(parts)

Preview what the LLM will see on step 0

In [12]:
demo = AgentState(goal="Find the capital of France.")
print("=== SYSTEM ===")
print(build_system_prompt())
print()
print("=== USER ===")
print(build_user_prompt(demo))

=== SYSTEM ===
You are a helpful agent that solves problems by calling tools.

You MUST respond with a single JSON object in one of these two shapes:

1. Tool call:
{"thought": "...", "action": {"type": "tool_call", "name": "<tool>", "args": {...}}}

2. Final answer:
{"thought": "...", "action": {"type": "final", "answer": "..."}}

Available tools:
- lookup(query: str) — Look up a fact by natural-language query.
- calculator(expression: str) — Evaluate an arithmetic expression. Supports + - * / and parentheses.

Rules:
- Output JSON only, no markdown, no backticks, no prose.
- Use a tool when you need external information or computation.
- Produce a final answer only when you have enough evidence.


=== USER ===
Goal: Find the capital of France.

What should we do next? Respond with JSON only.


## Validation — trust and verify

The LLM's JSON can be malformed in many ways: missing fields, wrong types, a tool name that doesn't exist, args that don't match the tool's schema. We catch each of those *before* executing anything.

Three principles:

1. **Fail fast** — catch bad args before spending the tool's latency/cost
2. **Return structured errors to the LLM** so it can self-correct on the next turn
3. **Halt only on fatal errors** (e.g., a prompt-injection attempt)

In [13]:
@dataclass
class ValidationResult:
    ok: bool
    error: str | None = None
    parsed: dict | None = None


def validate_proposal(raw_text: str) -> ValidationResult:
    # 1. Parse JSON
    try:
        data = json.loads(raw_text.strip())
    except json.JSONDecodeError as e:
        return ValidationResult(ok=False, error=f"Malformed JSON: {e}")

    # 2. Check top-level shape
    if not isinstance(data, dict) or "action" not in data:
        return ValidationResult(ok=False, error="Missing 'action' field.")

    action = data["action"]
    action_type = action.get("type")

    # 3. Validate by action type
    if action_type == "final":
        if not action.get("answer"):
            return ValidationResult(ok=False, error="Final action missing 'answer'.")
        return ValidationResult(ok=True, parsed=data)

    if action_type == "tool_call":
        name = action.get("name")
        if name not in TOOLS:
            known = sorted(TOOLS)
            return ValidationResult(
                ok=False,
                error=f"Unknown tool {name!r}. Known tools: {known}",
            )
        args = action.get("args", {})
        if not isinstance(args, dict):
            return ValidationResult(ok=False, error="args must be a JSON object.")
        tool_err = TOOLS[name].validate(args)
        if tool_err:
            return ValidationResult(ok=False, error=tool_err)
        return ValidationResult(ok=True, parsed=data)

    return ValidationResult(ok=False, error=f"Unknown action type: {action_type!r}")

## The control loop

In [14]:
MAX_CONSECUTIVE_ERRORS = 3

def run_agent(goal: str, llm_provider, budget: Budget | None = None) -> AgentState:
    state = AgentState(goal=goal, budget=budget or Budget())
    system = build_system_prompt()
    consecutive_errors = 0

    while state.status == "running" and state.budget.has_room():
        state.budget.steps_used += 1

        # 1. Build prompt
        user = build_user_prompt(state)

        # 2. Ask the LLM
        t0 = time.time()
        raw = llm_provider.complete(system=system, user=user)
        latency = int((time.time() - t0) * 1000)

        # 3. Validate
        result = validate_proposal(raw)
        if not result.ok:
            consecutive_errors += 1
            # Feed the error back as an observation so the LLM can self-correct
            state.trace.append(Step(
                index=state.budget.steps_used,
                thought="(parser)",
                action={"type": "invalid", "raw": raw[:200]},
                observation=f"VALIDATION_ERROR: {result.error}",
                latency_ms=latency,
            ))
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                state.status = "failed"
                state.final_answer = f"Too many validation errors: {result.error}"
            continue

        consecutive_errors = 0
        proposal = result.parsed
        action = proposal["action"]
        thought = proposal.get("thought", "")

        # 4. Execute
        if action["type"] == "final":
            state.trace.append(Step(
                index=state.budget.steps_used,
                thought=thought,
                action=action,
                observation="(final)",
                latency_ms=latency,
            ))
            state.status = "done"
            state.final_answer = action["answer"]
        else:  # tool_call
            tool = TOOLS[action["name"]]
            try:
                obs = tool.fn(**action["args"])
                state.budget.tool_calls_used += 1
            except Exception as e:
                obs = f"TOOL_ERROR: {type(e).__name__}: {e}"
            state.trace.append(Step(
                index=state.budget.steps_used,
                thought=thought,
                action=action,
                observation=obs,
                latency_ms=latency,
            ))

    if state.status == "running":
        state.status = "failed"
        state.final_answer = "Budget exhausted before goal was reached."

    return state


## Running the agent

Let's give it a compound task that needs two lookups and one calculation:

> *"Find the capital of France, look up its population, and compute what percentage that is of France's total population of 68 million."*

In [15]:
GOAL = (
    "Find the capital of France, look up its population, and compute what "
    "percentage that is of France's total population of 68 million."
)

In [16]:
final_state = run_agent(GOAL, llm)

In [17]:
print(f"Status: {final_state.status}")
print(f"Steps used: {final_state.budget.steps_used}")
print(f"Tool calls: {final_state.budget.tool_calls_used}")
print()
print("Final answer:")
print(f"  {final_state.final_answer}")

Status: done
Steps used: 10
Tool calls: 9

Final answer:
  The capital of France is Paris, which has a population of 2.1 million. This represents approximately 3.09% of France's total population of 68 million.


## Inspecting the trace

The trace is the most valuable artifact a harness produces. You can print it, diff it against a previous run, or replay it to debug. Below we pretty-print the trace and then show a simple visualization of per-step latency — the foundation of any observability story.

In [18]:
def print_trace(state: AgentState) -> None:
    for s in state.trace:
        print(f"--- Step {s.index}  ({s.latency_ms} ms) ---")
        print(f"  thought: {s.thought}")
        print(f"  action:  {json.dumps(s.action)}")
        print(f"  obs:     {s.observation}")
    print()

print_trace(final_state)


--- Step 1  (4939 ms) ---
  thought: I need to find the capital of France first. This is a straightforward fact that I can look up.
  action:  {"type": "tool_call", "name": "lookup", "args": {"query": "capital of France"}}
  obs:     Paris
--- Step 2  (1773 ms) ---
  thought: Now I know the capital is Paris. Next, I need to look up the population of Paris.
  action:  {"type": "tool_call", "name": "lookup", "args": {"query": "population of Paris"}}
  obs:     (no result for 'population of Paris')
--- Step 3  (3201 ms) ---
  thought: The lookup for 'population of Paris' didn't return a result. Let me try a more specific query about Paris's population, perhaps including 'city' or being more explicit.
  action:  {"type": "tool_call", "name": "lookup", "args": {"query": "Paris city population"}}
  obs:     (no result for 'Paris city population')
--- Step 4  (2723 ms) ---
  thought: The lookup tool hasn't returned results for Paris population queries. Let me try a different phrasing that mig

<center>
     <img src="data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>